## Setup

The following packages are required to run the analysis. If not already installed, the packages will be installed using `pip`.

In [ ]:
!pip install numpy
!pip install pandas
!pip install scipy
!pip install hmmlearn
!pip install statsmodels
!pip install tqdm
!pip install colorama

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from tqdm import tqdm
import statsmodels.api as sm
import scipy.stats as stats
import math
import pyBigWig
import glob
import requests
import warnings

# Suppress all warnings to keep the output clean
warnings.filterwarnings("ignore")

## Specify project directories

Define the paths to the data and results directories used in the project.

In [2]:
# Specify the directories containing data and results
data_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/data/'
results_path = '/oak/stanford/groups/mrivas/projects/wgs-constraint-llm/osthoag/wgs-constraint-llm/results/'

# Specify the paths to the specific data files
# https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_44/gencode.v44.chr_patch_hapl_scaff.basic.annotation.gtf.gz
gene_annotation_file_path = data_path + 'gencode.v44.basic.annotation.gtf.gz'
# https://github.com/bahlolab/Genes4Epilepsy/blob/3a8b2ebdb26daef453e5e806c7d1d42dd9fb3717/EpilepsyGenes_v2025-09.tsv
genes4epilepsy_file_path = data_path + 'EpilepsyGenes_v2025-09.tsv'
# https://epi25.broadinstitute.org/results
epi25_variants_file_path = data_path + 'epi25_variant_results.tsv.gz'
# https://genome.ucsc.edu/cgi-bin/hgTables?db=hg19&hgta_group=compGeno&hgta_track=allHg19RS_BW&hgta_table=allHg19RS_BW&hgta_doSchema=describe+table+schema
gerp_file_path = data_path + 'All_hg38_RS.bw'
# https://doi.org/10.6084/m9.figshare.27184245.v1
hmm_predictions_path = results_path + f'HMM_rgc_0.9_over20_chr2_predictions_rgc_wes.tsv.gz'
# https://zenodo.org/records/10813168/files/AlphaMissense_hg38.tsv.gz?download=1
alpha_missense_file_path = data_path + 'AlphaMissense_hg38.tsv.gz'

## Load Data
This section loads the gene annotation data from a GTF file and extracts relevant information such as gene ID, gene type, gene name, and transcript details. The data is then filtered to include only protein-coding regions.

In [ ]:
# Read the GTF file into a pandas DataFrame
gene_df = pd.read_csv(gene_annotation_file_path, sep='\t', comment='#', header=None, 
                      names=['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attribute'], 
                      dtype={'start': int, 'end': int})

# Extract 'gene_id' from attributes
gene_df['gene_id'] = gene_df['attribute'].str.extract(r'gene_id "(.*?)"')

# Extract 'gene_type' from attributes
gene_df['gene_type'] = gene_df['attribute'].str.extract(r'gene_type "(.*?)"')

# Extract 'gene_name' from attributes
gene_df['gene_name'] = gene_df['attribute'].str.extract(r'gene_name "(.*?)"')

# Extract 'transcript_id' from attributes
gene_df['transcript_id'] = gene_df['attribute'].str.extract(r'transcript_id "(.*?)"')

# Extract 'transcript' and 'num' from transcript_id
gene_df[['transcript', 'transcript_num']] = gene_df['transcript_id'].str.split('.', expand=True)

# Extract 'transcript_name' from attributes
gene_df['transcript_name'] = gene_df['attribute'].str.extract(r'transcript_name "(.*?)"')

# Drop the original attribute column
gene_df = gene_df.drop('attribute', axis=1)

# Filter rows for protein-coding regions
gene_df = gene_df[(gene_df['gene_type'] == 'protein_coding') & (gene_df['feature'] == 'CDS')]

# Standardize the gene_id by removing any version numbers (i.e., text after the dot)
gene_df['std_gene_id'] = gene_df['gene_id'].str.split('.').str[0]

# Display the DataFrame
gene_df

## Load and Filter Epilepsy Variant Data
This section reads in the epilepsy variant data, splits the chromosome and position information, filters out unwanted chromosomes and variant types, and merges it with the gene annotation data to include gene names.

In [ ]:
# Read the epilepsy variant data into a DataFrame
epi25_variant_results_df = pd.read_csv(epi25_variants_file_path, sep='\t')

# Split the locus into chromosome and position, and convert the position to an integer
epi25_variant_results_df[['chr', 'pos', 'ref', 'alt']] = epi25_variant_results_df['variant_id'].str.split(':', expand=True)
epi25_variant_results_df['pos'] = epi25_variant_results_df['pos'].astype(int)

# Filter out rows related to sex chromosomes and non-coding, synonymous, or undefined consequences
epi25_variant_results_df = epi25_variant_results_df[(epi25_variant_results_df['chr'] != "chrX") &
                                                    (epi25_variant_results_df['chr'] != "chrY") &
                                                    (epi25_variant_results_df['chr'] != "chrMT")]
epi25_variant_results_df = epi25_variant_results_df[(epi25_variant_results_df['consequence'] != "synonymous") & 
                                                    (epi25_variant_results_df['consequence'] != "non_coding") & 
                                                    (epi25_variant_results_df['consequence'] != "NA")]

# Merge the filtered variants with gene names from the gene annotation data
epi25_variant_results_df = pd.merge(epi25_variant_results_df, gene_df[['std_gene_id', 'gene_name']].drop_duplicates(), left_on='gene_id', right_on='std_gene_id', how='left').drop('std_gene_id', axis=1)

# Display the DataFrame
epi25_variant_results_df

## Load Constraint Predictions
Load saved HMM predictions for constraint data, which will be used in building the unified model.

In [ ]:
# Load in saved predictions for the constraint dataset
constraint_predictions_df = pd.read_csv(constraint_predictions_file_path, sep='\t')

# Display the DataFrame
constraint_predictions_df

In [ ]:
# GERP RS annotation.
#
# This cell previously inlined a bigWig lookup that indexed a 0-based value
# vector with 1-based HMM positions, shifting every score one base downstream.
# The lookup now lives in src/wgs_constraint/gerp.py, so the three notebooks
# that need it cannot drift apart again; the coordinate convention is an
# explicit argument there rather than an assumption.
#
# Loading the predictions is kept here because later cells use `pred`.
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd

from wgs_constraint import annotate_with_gerp

# --- load HMM predictions (once) ---
pred = pd.read_csv(hmm_predictions_path, sep="\t", dtype={"chr": "string"})
if "position" in pred.columns and "pos" not in pred.columns:
    pred = pred.rename(columns={"position": "pos"})
pred["pos"] = pred["pos"].astype(np.int64)
pred["chr"] = pred["chr"].astype("string")
pred_cols = pred.columns.tolist()

merged_constraint_df = annotate_with_gerp(
    pred,
    gerp_file_path,
    position_base=1,      # HMM positions are 1-based
)
merged_constraint_df["chr"] = merged_constraint_df["chr"].astype("category")
merged_constraint_df


## Load Missense Pathogenicity Data
Load and preprocess missense pathogenicity data, including extracting transcript details and renaming columns for consistency.

In [ ]:
# Read the missense pathogenicity data into a pandas DataFrame
alpha_missense_df = pd.read_csv(alpha_missense_file_path, sep='\t', header=3)

# Rename columns to standard labels
alpha_missense_df.rename(columns={"#CHROM": "chr", "POS": "pos", "REF": "ref", "ALT": 'alt'}, inplace=True)

# Extract 'transcript' and 'transcript_num' from transcript_id
alpha_missense_df[['transcript', 'transcript_num']] = alpha_missense_df['transcript_id'].str.split('.', expand=True)

# Display the DataFrame
alpha_missense_df

## Filter Variants for Analysis
Further filtering of variants is done to include only those with a total allele count (ac_ctrl + ac_case) of 5 or less, and with non-zero allele numbers in both cases and controls.

In [ ]:
# Copy results df before applying model-specific transformations
variants_df = epi25_variant_results_df

# Filter variants where the sum of allele counts in cases and controls is less than or equal to 5
# and where both case and control allele numbers are non-zero
variants_df = variants_df[(variants_df['ac_ctrl'] + variants_df['ac_case'] <= 5) &
                          (variants_df['an_case'] > 0) &
                          (variants_df['an_ctrl'] > 0)]

# Calculate effect size and variance for variants
ALT_AJ = variants_df['ac_case']
ALT_ExAC = variants_df['ac_ctrl']
REF_AJ = variants_df['an_case'] - variants_df['ac_case']
REF_ExAC = variants_df['an_ctrl'] - variants_df['ac_ctrl']

# Logarithmic transformation of effect size calculation
variants_df['effect_size'] = np.log(((0.5 + ALT_AJ) * (0.5 + REF_ExAC)) / ((0.5 + REF_AJ) * (0.5 + ALT_ExAC)))

# Variance calculation for the effect size
variants_df['var_effect_size'] = (1 / (0.5 + REF_AJ) + 1 / (0.5 + REF_ExAC) + 1 / (0.5 + ALT_AJ) + 1 / (0.5 + ALT_ExAC))

# Add indicator variable for pLoF variants
variants_df['pLoF_ind'] = (variants_df['consequence'] == "pLoF").astype('int32')

# Add indicator variable for missense variants
variants_df['missense_ind'] = ((variants_df['consequence'] == "damaging_missense") | (variants_df['consequence'] == "other_missense")).astype('int32')

# Display the DataFrame
variants_df

In [ ]:
# Subset the constraint predictions to include only relevant columns
constraint_subset_df = merged_constraint_df[['chr', 'pos', 'prob_0', 'GERP_RS']]

# Subset the AlphaMissense predictions to include only relevant columns
am_subset_df = alpha_missense_df[['chr', 'pos', 'ref', 'alt', 'am_pathogenicity']]

# Subset the pLoF dataframe to include only useful columns
variants_subset_df = variants_df[['chr', 'pos', 'ref', 'alt', 'gene_id', 'gene_name', 'group', 'ac_case', 'an_case', 'ac_ctrl', 'an_ctrl', 'pLoF_ind', 'missense_ind', 'effect_size', 'var_effect_size']]

# Merge constraint predictions, pathogenicity predictions, and pLoF variants based on chromosome and position
constraint_pathogenicity_pLoF_df = pd.merge(constraint_subset_df, pd.merge(variants_subset_df, am_subset_df, on=['chr', 'pos', 'ref', 'alt'], how='left'), on=['chr', 'pos'], how='inner')

# Save the merged dataframe to a compressed CSV file for further analysis
constraint_pathogenicity_pLoF_df.to_csv(results_path + f"constraint_gerp_am_epi25_variants.tsv.gz", index=False, compression='gzip', sep='\t')

# Display the DataFrame
constraint_pathogenicity_pLoF_df

## Build Unified Constraint, Pathogenicity, and pLoF Model
This section creates a unified model that integrates constraint predictions, pathogenicity predictions, and loss-of-function (pLoF) indicators to assess the association with epilepsy.

In [ ]:
# Read the data from the file
input_df = pd.read_csv(results_path + f'constraint_gerp_am_epi25_variants.tsv.gz', sep='\t')

# frequency threshold
# input_df = input_df[input_df['AF'] <= 0.05]

# Ensure probabilities are not exactly 1 or 0
epsilon = 1e-2
input_df['prob_0'] = np.clip(input_df['prob_0'], epsilon, 1 - epsilon)
input_df['am_pathogenicity'] = np.clip(input_df['am_pathogenicity'], epsilon, 1 - epsilon)

# Impute missing values with column means
input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']] = input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']].fillna(input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']].mean())
input_df['pLoF_ind'] = input_df['pLoF_ind'].fillna(0)
input_df['missense_ind'] = input_df['missense_ind'].fillna(0)

# Apply log transformations
# input_df[['log_constraint', 'log_gerp', 'log_pathogenicity']] = -np.log1p(-(input_df[['prob_0', 'GERP_RS', 'am_pathogenicity']]))
input_df[['log_constraint', 'log_pathogenicity']] = -np.log1p(-(input_df[['prob_0', 'am_pathogenicity']]))

# Remove rows with missing effect_size or var_effect_size
input_df = input_df.dropna(subset=['effect_size', 'var_effect_size'])
input_df = input_df[input_df['var_effect_size'] != 0]

# Display table of inputs to meta-regression model
input_df

In [ ]:
from statsmodels.regression.linear_model import WLS
from statsmodels.tools.tools import add_constant
    
# Group data by gene
grouped_gene_data = input_df.groupby(['gene_id', 'gene_name', 'group'])

# Initialize lists to store results
meta_model_results = []

# Loop over each gene group and build a meta-regression model
for gene_key, gene_data in tqdm(grouped_gene_data, desc="Processing genes", unit="gene"):
    gene_id, gene_name, group = gene_key

    if gene_data[['log_constraint', 'GERP_RS', 'log_pathogenicity', 'pLoF_ind', 'missense_ind', 'effect_size', 'var_effect_size']].isnull().any().any():
        continue

    # Meta-regression model for the gene
    X = add_constant(gene_data[['log_constraint', 'GERP_RS', 'log_pathogenicity', 'pLoF_ind', 'missense_ind']])
    y = gene_data['effect_size']
    weights = 1 / gene_data['var_effect_size']

    try:
        model = WLS(y, X, weights=weights, missing='drop').fit()

        # Append relevant results to the meta_model_results list
        meta_model_results.append({
            'gene_id': gene_id,
            'gene_name': gene_name,
            'group': group,
            'n_variants': len(gene_data),
            'p_constraint': model.pvalues['log_constraint'],
            'p_gerp': model.pvalues['GERP_RS'],
            'p_pathogenicity': model.pvalues['log_pathogenicity'],
            'p_pLoF': model.pvalues['pLoF_ind'],
            'p_missense': model.pvalues['missense_ind'],
    #         'p_const': model.pvalues['const'],
            'p_unified': model.f_pvalue
        })
        
    except Exception as e:
#         print(f"Error processing {gene_key}: {str(e)}")
        pass

# Create a DataFrame from the results
unified_model_df = pd.DataFrame(meta_model_results)

# Save the results to a compressed CSV file
unified_model_df.to_csv(results_path + "epilepsy_unified_model_pvalues.tsv", index=False, sep='\t')

# Display the contents of the DataFrame
unified_model_df

In [ ]:
import glob
import numpy as np
import pandas as pd

# -------------------
# Settings
# -------------------
min_variants = 25
unified_path = results_path + "epilepsy_unified_model_pvalues.tsv"
out_path = results_path + f"unified_epi25_g4e_merged_n{min_variants}.tsv.gz"

eps = 1e-300  # for safe -log10 clipping

# -------------------
# Load Genes4Epilepsy
# -------------------
g4e = pd.read_csv(genes4epilepsy_file_path, sep="\t", dtype=str)

candidate_cols = [c for c in g4e.columns if any(k in c.lower() for k in ["gene", "symbol", "hgnc"])]
if not candidate_cols:
    raise ValueError(f"Could not find a gene symbol column in {genes4epilepsy_file_path}. Columns: {list(g4e.columns)}")

preferred = [c for c in candidate_cols if c.lower() in ["gene", "gene_name", "gene symbol", "genesymbol", "hgnc", "hgnc_symbol", "symbol"]]
gene_col = preferred[0] if preferred else candidate_cols[0]

g4e_set = set(g4e[gene_col].dropna().astype(str).str.strip().str.upper().tolist())
print(f"Loaded Genes4Epilepsy set: {len(g4e_set):,} unique symbols (using column '{gene_col}').")

# -------------------
# Load unified model results
# -------------------
unified_model_df = pd.read_csv(unified_path, sep="\t")

# Filter by variants
unified_model_df = unified_model_df[unified_model_df["n_variants"] >= min_variants].copy()

# Coerce p-values safely
unified_model_df["p_unified"] = pd.to_numeric(unified_model_df["p_unified"], errors="coerce")

# Add G4E membership (by gene_name symbol)
unified_model_df["gene_name_upper"] = unified_model_df["gene_name"].astype(str).str.strip().str.upper()
unified_model_df["G4E"] = unified_model_df["gene_name_upper"].isin(g4e_set)

# -------------------
# Load + join Epi25 per group, concat
# -------------------
merged_list = []

for group in unified_model_df["group"].dropna().unique():
    file_pattern = data_path + f"{group}_results_2023_12_30_14_36_*.csv"
    files = glob.glob(file_pattern)

    if not files:
        print(f"No Epi25 file found for group {group} (pattern: {file_pattern})")
        continue

    epi25_df = pd.read_csv(files[0], sep=",")

    # Coerce Epi25 p-values safely
    epi25_df["Damaging Missense p‑val"] = pd.to_numeric(epi25_df["Damaging Missense p‑val"], errors="coerce")
    epi25_df["PTV p‑val"] = pd.to_numeric(epi25_df["PTV p‑val"], errors="coerce")

    # Join on gene_id <-> Gene
    merged = pd.merge(
        unified_model_df[unified_model_df["group"] == group],
        epi25_df,
        left_on="gene_id",
        right_on="Gene",
        how="inner"
    ).copy()

    # Epi25 min p-value (robust to NaNs)
    merged["epi25_min_p"] = merged[["Damaging Missense p‑val", "PTV p‑val"]].min(axis=1, skipna=True)

    # -log10 p-values (clip away 0)
    merged["minus_log_unified"] = -np.log10(np.clip(merged["p_unified"].to_numpy(dtype=float), eps, 1.0))
    merged["minus_log_epi25"] = -np.log10(np.clip(merged["epi25_min_p"].to_numpy(dtype=float), eps, 1.0))

    merged_list.append(merged)

merged_df = pd.concat(merged_list, ignore_index=True) if merged_list else pd.DataFrame()

# Keep finite values for plotting columns (but do NOT drop rows based on raw p-values)
merged_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# -------------------
# Save merged dataframe
# -------------------
merged_df.to_csv(out_path, sep="\t", index=False, compression="gzip")

In [ ]:
# =========================
# Load merged unified+epi25+g4e output and:
#   (1) Recreate the 2x2 heatmap grid by group using minus_log_* columns
#   (2) Display a significance-filtered table (either Unified or Epi25 crosses thr)
# =========================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# -------------------
# Settings
# -------------------
min_variants = 25
merged_path = results_path + f"unified_epi25_g4e_merged_n{min_variants}.tsv.gz"

cand_thr = 3.4e-7
sugg_thr = 1e-4   # retained if you still want it later
num_bins = 10

# -------------------
# Load merged dataframe
# -------------------
merged_df = pd.read_csv(merged_path, sep="\t", compression="gzip")

# Coerce numeric columns used below (robust if read as object)
for c in ["p_unified", "epi25_min_p", "minus_log_unified", "minus_log_epi25"]:
    if c in merged_df.columns:
        merged_df[c] = pd.to_numeric(merged_df[c], errors="coerce")

# Drop rows without plotting values (these were already filtered in the merge step,
# but keep it safe in case the saved file format changes later)
plot_df = merged_df.dropna(subset=["minus_log_unified", "minus_log_epi25", "group"]).copy()

# -------------------
# Recreate merged_by_group dict expected by plotting code
# -------------------
merged_by_group = {
    g: df.copy()
    for g, df in plot_df.groupby("group")
}

# -------------------
# 1) HEATMAP PLOT (same plot, now driven by merged_df on disk)
# -------------------
groups = list(merged_by_group.keys())

# If you truly expect exactly 4 groups, uncomment to enforce:
# assert len(groups) == 4, f"Expected 4 groups, found {len(groups)}: {groups}"

global_max = 0
for df in merged_by_group.values():
    global_max = max(global_max, df["minus_log_epi25"].max(), df["minus_log_unified"].max())
if not np.isfinite(global_max):
    global_max = 1

limit = global_max + 1
x_edges = np.linspace(0, limit, num_bins + 1)
y_edges = np.linspace(0, limit, num_bins + 1)

fig = plt.figure(figsize=(12, 10))

L, R = 0.08, 0.88
B, T = 0.08, 0.92
W = R - L
H = T - B

side = min(W / 2, H / 2)
grid_W = 2 * side
grid_H = 2 * side
grid_L = L + (W - grid_W) / 2
grid_B = B + (H - grid_H) / 2

positions = {
    0: [grid_L + 0 * side, grid_B + 1 * side, side, side],
    1: [grid_L + 1 * side, grid_B + 1 * side, side, side],
    2: [grid_L + 0 * side, grid_B + 0 * side, side, side],
    3: [grid_L + 1 * side, grid_B + 0 * side, side, side],
}
axes = [fig.add_axes(positions[i]) for i in range(4)]
mappable = None

for ax, group in zip(axes, groups):
    df = merged_by_group[group]

    h = ax.hist2d(
        df["minus_log_epi25"],
        df["minus_log_unified"],
        bins=[x_edges, y_edges],
        cmap="Blues",
        norm=LogNorm()
    )
    mappable = h[3]

    ax.plot([0, limit], [0, limit], color="red", linestyle="--", linewidth=1)

    annot_df = df[
        (np.abs(df["minus_log_epi25"] - df["minus_log_unified"]) > 2) &
        (df[["minus_log_epi25", "minus_log_unified"]].max(axis=1) > 6)
    ].copy()

    annot_df["x_bin_idx"] = np.digitize(annot_df["minus_log_epi25"], bins=x_edges) - 1
    annot_df["y_bin_idx"] = np.digitize(annot_df["minus_log_unified"], bins=y_edges) - 1

    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    stack_offset = limit * 0.02

    # gene_name in your unified df; fall back to whatever exists
    name_col = "gene_name" if "gene_name" in df.columns else ("Gene Name" if "Gene Name" in df.columns else None)

    if name_col is not None:
        for (x_idx, y_idx), gdf in annot_df.groupby(["x_bin_idx", "y_bin_idx"]):
            if 0 <= x_idx < len(x_centers) and 0 <= y_idx < len(y_centers):
                x_center = x_centers[x_idx]
                y_center = y_centers[y_idx]
                total_h = (len(gdf) - 1) * stack_offset

                gdf_sorted = gdf.sort_values(by="minus_log_unified")
                for i, (_, row) in enumerate(gdf_sorted.iterrows()):
                    ax.text(
                        x_center,
                        y_center - total_h / 2 + i * stack_offset,
                        str(row[name_col]),
                        fontsize=6,
                        ha="center"
                    )

    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)

    ax.text(
        0.97 * limit, 0.03 * limit, "Group: " + str(group),
        fontsize=12, fontweight="bold",
        ha="right", va="bottom"
    )

# Remove interior tick labels
axes[0].set_xticklabels([])
axes[1].set_xticklabels([])
axes[1].set_yticklabels([])
axes[3].set_yticklabels([])

fig.suptitle(
    "Unified model vs Epi25 p-values across epilepsy groups",
    fontsize=16,
    fontweight="bold",
    y=0.97
)
fig.supxlabel(r"Epi25 $-log_{10}(p)$: min(Missense, PTV) ", fontsize=14)
fig.supylabel(r"Unified model $-log_{10}(p)$", fontsize=14)

cbar_ax = fig.add_axes([0.91, grid_B, 0.02, grid_H])
cbar = fig.colorbar(mappable, cax=cbar_ax)
cbar.set_label("Log-scaled count")

plt.savefig(
    results_path + f"Figure_epi25_heatmap_2x2_grid_square_nogap_from_merged_n{min_variants}.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [59]:
import numpy as np
import pandas as pd

# -------------------
# Settings
# -------------------
min_variants = 25
merged_path = results_path + f"unified_epi25_g4e_merged_n{min_variants}.tsv.gz"
out_csv = results_path + f"table_A1_epilepsy_full_p_value_comparison_n{min_variants}.csv"

# -------------------
# Load merged unified+epi25(+G4E already included) dataframe
# -------------------
full_pub_df = pd.read_csv(merged_path, sep="\t", compression="gzip")

# -------------------
# Build display / export table (same logic as before, but no G4E loading/merging)
# -------------------
# Find Epi25 columns robustly (handles unicode hyphen variants in column names)
dm_col = next((c for c in full_pub_df.columns if "Damaging Missense" in c and "p" in c), None)
ptv_col = next((c for c in full_pub_df.columns if "PTV" in c and "p" in c), None)
if dm_col is None or ptv_col is None:
    raise ValueError(f"Could not find Epi25 p-value columns in merged file. Columns: {list(full_pub_df.columns)}")

# Subset columns (match your prior comparison_df construction; keep G4E if present)
base_cols = [
    "gene_id", "gene_name", "group", "n_variants",
    "p_constraint", "p_gerp", "p_pathogenicity", "p_pLoF", "p_missense", "p_unified",
    dm_col, ptv_col
]
if "G4E" in full_pub_df.columns:
    base_cols.append("G4E")

# Keep only columns that actually exist (defensive)
base_cols = [c for c in base_cols if c in full_pub_df.columns]
comparison_df = full_pub_df[base_cols].copy()

# Rename columns to match your previous exported CSV
column_names = {
    "gene_id": "Gene ID",
    "gene_name": "Gene Name",
    "group": "Group",
    "n_variants": "# of Variants",
    "p_constraint": "Constraint p-value",
    "p_gerp": "GERP p-value",
    "p_pathogenicity": "Pathogenicity p-value",
    "p_pLoF": "pLoF p-value",
    "p_missense": "Missense p-value",
    dm_col: "Epi25 DM p-value",
    ptv_col: "Epi25 PTV p-value",
    "p_unified": "Unified Model p-value",
}
comparison_df.rename(columns=column_names, inplace=True)

# Coerce p-values to numeric safely
for c in [
    "Constraint p-value", "GERP p-value", "Pathogenicity p-value",
    "pLoF p-value", "Missense p-value", "Unified Model p-value",
    "Epi25 DM p-value", "Epi25 PTV p-value"
]:
    if c in comparison_df.columns:
        comparison_df[c] = pd.to_numeric(comparison_df[c], errors="coerce")

# Robust Epi25 min p-value (handles NaNs by taking the other value; NaN only if both NaN)
comparison_df["Epi25 min p-value"] = comparison_df[["Epi25 DM p-value", "Epi25 PTV p-value"]].min(axis=1, skipna=True)

# Save
comparison_df.to_csv(out_csv, index=False)

In [ ]:
# Read saved results
cand_thr = 3.4e-7
sugg_thr = 1e-4
min_variants = 25
comparison_df = pd.read_csv(results_path + f"table_A1_epilepsy_full_p_value_comparison_n{min_variants}.csv")

# Filter the DataFrame for significant p-values or specific gene names for closer examination
sugg_mask = (comparison_df['Unified Model p-value'] < sugg_thr)
cand_mask = (comparison_df['Unified Model p-value'] < cand_thr)
epi25_mask = (comparison_df["Epi25 min p-value"] < cand_thr)
g4e_mask = (comparison_df['G4E'].astype(bool))

# Move "Unified Model p-value" to the last column
disp_cols = ["Gene Name", "Group", "# of Variants",
        "Constraint p-value", "GERP p-value", "Pathogenicity p-value", "pLoF p-value", "Missense p-value",
        "Epi25 DM p-value", "Epi25 PTV p-value", "Unified Model p-value"]

# Display the filtered DataFrame sorted by the unified p-value
pd.set_option('display.max_rows', 300)
(
    comparison_df[cand_mask | epi25_mask]
    .sort_values('Unified Model p-value')[disp_cols]
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)

In [ ]:
gene_df = (
    comparison_df
    .assign(_g4e=comparison_df["G4E"].astype(bool))
    .groupby("Gene ID", as_index=False)
    .agg({
        "Gene Name": "first",
        "_g4e": "max",
        "Unified Model p-value": "min",
        "Epi25 min p-value": "min",
    })
)

gene_df["meta_sig"] = gene_df["Unified Model p-value"] < cand_thr
gene_df["epi25_sig"] = gene_df["Epi25 min p-value"] < cand_thr

def summarize(subdf, label):
    total = subdf.shape[0]
    meta = int(subdf["meta_sig"].sum())
    epi = int(subdf["epi25_sig"].sum())
    both = int((subdf["meta_sig"] & subdf["epi25_sig"]).sum())
    meta_only = int((subdf["meta_sig"] & ~subdf["epi25_sig"]).sum())
    epi_only = int((~subdf["meta_sig"] & subdf["epi25_sig"]).sum())
    neither = int((~subdf["meta_sig"] & ~subdf["epi25_sig"]).sum())
    return {
        "subset": label,
        "# genes": total,
        "# meta_sig": meta,
        "# epi25_sig": epi,
        "# both_sig": both,
        "# meta_only": meta_only,
        "# epi25_only": epi_only,
        "# neither": neither,
    }

summary = pd.DataFrame([
    summarize(gene_df[gene_df["_g4e"]], "In G4E"),
    summarize(gene_df[~gene_df["_g4e"]], "Not in G4E"),
    summarize(gene_df, "All"),
])

summary.style.hide(axis='index')

In [ ]:
disp_cols = ["Gene Name", "Group", "# of Variants",
        "Constraint p-value", "GERP p-value", "Pathogenicity p-value", "pLoF p-value", "Missense p-value",
        "Epi25 DM p-value", "Epi25 PTV p-value", "Unified Model p-value"]
(
    comparison_df[cols][sugg_mask & g4e_mask & ~cand_mask]
    .sort_values('Unified Model p-value')[disp_cols]
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)

In [ ]:
disp_cols = ["Gene Name", "Group", "# of Variants", "G4E",
        "Constraint p-value", "GERP p-value", "Pathogenicity p-value", "pLoF p-value", "Missense p-value",
        "Epi25 DM p-value", "Epi25 PTV p-value", "Unified Model p-value"]
# Display the filtered DataFrame sorted by the unified p-value
pd.set_option('display.max_rows', 300)
(
    comparison_df[sugg_mask]
    .sort_values('Unified Model p-value')[disp_cols]
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# -------------------
# Settings
# -------------------
num_bins = 10
groups = list(merged_by_group.keys())  # expects 4 groups

# -------------------
# Global symmetric bounds
# -------------------
global_max = 0
for df in merged_by_group.values():
    global_max = max(global_max, df["minus_log_epi25"].max(), df["minus_log_unified"].max())
if not np.isfinite(global_max):
    global_max = 1

limit = global_max + 1
x_edges = np.linspace(0, limit, num_bins + 1)
y_edges = np.linspace(0, limit, num_bins + 1)

# -------------------
# Figure + manual layout
# -------------------
fig = plt.figure(figsize=(12, 10))

# Layout region reserved for the 2x2 grid (leave room on right for colorbar)
L, R = 0.08, 0.88
B, T = 0.08, 0.92

W = R - L
H = T - B

# Make panels perfectly square and abutting: pick the largest square size that fits 2x2
side = min(W / 2, H / 2)

# Re-center the square grid in the available region
grid_W = 2 * side
grid_H = 2 * side
grid_L = L + (W - grid_W) / 2
grid_B = B + (H - grid_H) / 2

# Axes positions: (col, row) with row 0 = bottom, row 1 = top
positions = {
    0: [grid_L + 0 * side, grid_B + 1 * side, side, side],  # top-left
    1: [grid_L + 1 * side, grid_B + 1 * side, side, side],  # top-right
    2: [grid_L + 0 * side, grid_B + 0 * side, side, side],  # bottom-left
    3: [grid_L + 1 * side, grid_B + 0 * side, side, side],  # bottom-right
}

axes = [fig.add_axes(positions[i]) for i in range(4)]

mappable = None

# -------------------
# Plot each panel
# -------------------
for ax, group in zip(axes, groups):
    df = merged_by_group[group]

    h = ax.hist2d(
        df["minus_log_epi25"],
        df["minus_log_unified"],
        bins=[x_edges, y_edges],
        cmap="Blues",
        norm=LogNorm()
    )
    mappable = h[3]

    # Diagonal
    ax.plot([0, limit], [0, limit], color="red", linestyle="--", linewidth=1)

    # Annotation selection
    annot_df = df[
        (np.abs(df["minus_log_epi25"] - df["minus_log_unified"]) > 2) &
        (df[["minus_log_epi25", "minus_log_unified"]].max(axis=1) > 6)
    ].copy()

    annot_df["x_bin_idx"] = np.digitize(annot_df["minus_log_epi25"], bins=x_edges) - 1
    annot_df["y_bin_idx"] = np.digitize(annot_df["minus_log_unified"], bins=y_edges) - 1

    x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
    y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
    stack_offset = limit * 0.02

    for (x_idx, y_idx), gdf in annot_df.groupby(["x_bin_idx", "y_bin_idx"]):
        if 0 <= x_idx < len(x_centers) and 0 <= y_idx < len(y_centers):
            x_center = x_centers[x_idx]
            y_center = y_centers[y_idx]
            total_h = (len(gdf) - 1) * stack_offset

            gdf_sorted = gdf.sort_values(by="minus_log_unified")
            for i, (_, row) in enumerate(gdf_sorted.iterrows()):
                ax.text(
                    x_center,
                    y_center - total_h / 2 + i * stack_offset,
                    row["gene_name"],
                    fontsize=6,
                    ha="center"
                )

    # Square, shared limits
    ax.set_xlim(0, limit)
    ax.set_ylim(0, limit)

    # Group label bottom-right
    ax.text(
        0.97 * limit, 0.03 * limit, "Group: " + group,
        fontsize=12, fontweight="bold",
        ha="right", va="bottom"
    )

# -------------------
# Make it look like a single grid: remove interior tick labels
# -------------------
# Keep x tick labels only on bottom row (axes[2], axes[3])
axes[0].set_xticklabels([])
axes[1].set_xticklabels([])

# Keep y tick labels only on left column (axes[0], axes[2])
axes[1].set_yticklabels([])
axes[3].set_yticklabels([])

# -------------------
# Shared title + axis labels
# -------------------
fig.suptitle(
    "Unified model vs Epi25 p-values across epilepsy groups",
    fontsize=16,
    fontweight="bold",
    y=0.97
)
fig.supxlabel(r"Epi25 $-log_{10}(p)$: min(Missense, PTV) ", fontsize=14)
fig.supylabel(r"Unified model $-log_{10}(p)$", fontsize=14)

# -------------------
# Colorbar (fully outside, no overlap)
# -------------------
cbar_ax = fig.add_axes([0.91, grid_B, 0.02, grid_H])  # aligned to grid height
cbar = fig.colorbar(mappable, cax=cbar_ax)
cbar.set_label("Log-scaled count")

plt.savefig(
    results_path + "Figure_epi25_heatmap_2x2_grid_square_nogap.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [7]:
# Combine Epi25 data with our predictions and genes4epilepsy database for full comparison of results
full_pub_df = pd.concat(pub_dfs)
comparison_df = full_pub_df[['gene_id', 'gene_name', 'group', 'n_variants', 'p_constraint', 'p_gerp', 'p_pathogenicity', 'p_pLoF', 'p_missense', 'p_unified', 'Damaging Missense p‑val', 'PTV p‑val']]
column_names = {
    'gene_id': 'Gene ID',
    'gene_name': 'Gene Name',
    'group': 'Group',
    'n_variants': '# of Variants',
    'p_constraint': 'Constraint p-value',
    'p_gerp': 'GERP p-value',
    'p_pathogenicity': 'Pathogenicity p-value',
    'p_pLoF': 'pLoF p-value',
    'p_missense': 'Missense p-value',
    'p_unified': 'Unified Model p-value',
    'Damaging Missense p‑val': 'Epi25 DM p-value',
    'PTV p‑val': 'Epi25 PTV p-value'
}
comparison_df.rename(columns=column_names, inplace=True)

# --- Load Genes4Epilepsy gene set ---
g4e = pd.read_csv(genes4epilepsy_file_path, sep="\t", dtype=str)
candidate_cols = [c for c in g4e.columns if any(k in c.lower() for k in ["gene", "symbol", "hgnc"])]
if not candidate_cols:
    raise ValueError(f"Could not find a gene symbol column in {genes4epilepsy_file_path}. Columns: {list(g4e.columns)}")
preferred = [c for c in candidate_cols if c.lower() in ["gene", "gene_name", "gene symbol", "genesymbol", "hgnc", "hgnc_symbol", "symbol"]]
gene_col = preferred[0] if preferred else candidate_cols[0]
genes4e_set = set(g4e[gene_col].dropna().astype(str).str.strip().str.upper().tolist())

# --- Add G4E membership column ---
comparison_df["G4E"] = comparison_df["Gene Name"].astype(str).str.strip().str.upper().isin(genes4e_set)

# --- Coerce p-values to numeric safely ---
comparison_df["Unified Model p-value"] = pd.to_numeric(comparison_df["Unified Model p-value"], errors="coerce")
comparison_df["Epi25 DM p-value"] = pd.to_numeric(comparison_df["Epi25 DM p-value"], errors="coerce")
comparison_df["Epi25 PTV p-value"] = pd.to_numeric(comparison_df["Epi25 PTV p-value"], errors="coerce")

# --- Robust Epi25 min p-value (handles NaNs by taking the other value; NaN only if both NaN) ---
comparison_df["Epi25 min p-value"] = comparison_df[["Epi25 DM p-value", "Epi25 PTV p-value"]].min(axis=1, skipna=True)

comparison_df.to_csv(results_path + f"table_A1_epilepsy_full_p_value_comparison_n{min_variants}.csv", index=False)

In [ ]:
# Read saved results
cand_thr = 3.4e-7
sugg_thr = 1e-4
min_variants = 25
comparison_df = pd.read_csv(results_path + f"table_A1_epilepsy_full_p_value_comparison_n{min_variants}.csv")

# Filter the DataFrame for significant p-values or specific gene names for closer examination
sugg_mask = (comparison_df['Unified Model p-value'] < sugg_thr)
cand_mask = (comparison_df['Unified Model p-value'] < cand_thr)
epi25_mask = (comparison_df["Epi25 min p-value"] < cand_thr)
g4e_mask = (comparison_df['G4E'].astype(bool))

# Move "Unified Model p-value" to the last column
col_last = "Unified Model p-value"
cols = [c for c in comparison_df.columns if c != col_last] + [col_last]

# Display the filtered DataFrame sorted by the unified p-value
pd.set_option('display.max_rows', 300)
(
    comparison_df[cols][cand_mask | epi25_mask]
    .drop(columns=['Gene ID', 'G4E', 'Epi25 min p-value'])
    .sort_values('Unified Model p-value')
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)

In [ ]:
(
    comparison_df[sugg_mask & g4e_mask & ~cand_mask]
    .drop(columns=['Gene ID', 'G4E', 'Epi25 min p-value'])
    .sort_values('Unified Model p-value')
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)

In [ ]:
# Display the filtered DataFrame sorted by the unified p-value
pd.set_option('display.max_rows', 500)
(
    comparison_df[sugg_mask]
    .drop(columns=['Gene ID', 'Epi25 min p-value'])  # drop the Gene ID column
    .sort_values('Unified Model p-value')
    .style
    .format({col: "{:.2e}" for col in comparison_df.columns if 'p-value' in col})
    .hide(axis='index')  # hide the index column
)

## Compare p-values with vs without various moderators

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from statsmodels.regression.linear_model import WLS
from statsmodels.tools.tools import add_constant

# -------------------
# Settings
# -------------------
full_model_fname = "epilepsy_unified_model_pvalues.tsv"  # already generated earlier
moderators = ["log_constraint", "GERP_RS", "log_pathogenicity", "pLoF_ind", "missense_ind"]
base_predictors = moderators.copy()

min_variants = 25  # match plotting filter later

# -------------------
# Load FULL model results (do not rerun)
# -------------------
# Prefer in-memory unified_model_df if present; else load from disk.
if "unified_model_df" in globals() and isinstance(unified_model_df, pd.DataFrame) and len(unified_model_df) > 0:
    full_df = unified_model_df.copy()
else:
    full_df = pd.read_csv(results_path + full_model_fname, sep="\t")

# Ensure unified p-value numeric
full_df["p_unified"] = pd.to_numeric(full_df["p_unified"], errors="coerce")

# -------------------
# Helper: run one reduced model
# -------------------
def run_meta_regression_with_predictors(
    input_df: pd.DataFrame,
    predictors: list[str],
    desc: str,
) -> pd.DataFrame:
    required_cols = predictors + ["effect_size", "var_effect_size"]

    grouped = input_df.groupby(["gene_id", "gene_name", "group"])
    rows = []

    for gene_key, gene_data in tqdm(grouped, desc=desc, unit="gene"):
        gene_id, gene_name, group = gene_key

        if gene_data[required_cols].isnull().any().any():
            continue

        X = add_constant(gene_data[predictors])
        y = gene_data["effect_size"]
        weights = 1.0 / gene_data["var_effect_size"]

        try:
            model = WLS(y, X, weights=weights, missing="drop").fit()
            out = {
                "gene_id": gene_id,
                "gene_name": gene_name,
                "group": group,
                "n_variants": len(gene_data),
                "p_unified": model.f_pvalue,
            }
            rows.append(out)
        except Exception:
            pass

    return pd.DataFrame(rows)

# -------------------
# Run and write all reduced models (FULL minus one moderator)
# -------------------
reduced_paths = {}  # moderator -> path

for dropped in moderators:
    reduced_predictors = [p for p in base_predictors if p != dropped]
    out_fname = f"epilepsy_unified_model_pvalues_MINUS_{dropped}.tsv"
    out_path = results_path + out_fname
    reduced_paths[dropped] = out_path

    reduced_df = run_meta_regression_with_predictors(
        input_df=input_df,
        predictors=reduced_predictors,
        desc=f"Meta-regression (FULL minus {dropped})",
    )

    reduced_df.to_csv(out_path, index=False, sep="\t")
    print(f"Wrote: {out_path}  (rows={len(reduced_df):,})")

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from statsmodels.regression.linear_model import WLS
from statsmodels.tools.tools import add_constant

# ============================================================
# Run an alternative model that EXCLUDES pLoF_ind and missense_ind
# (i.e., uses only: log_constraint, GERP_RS, log_pathogenicity)
# ============================================================

# -------------------
# Settings
# -------------------
alt_predictors = ["log_constraint", "GERP_RS", "log_pathogenicity"]
out_fname = "epilepsy_unified_model_pvalues_NO_pLoF_NO_missense.tsv"
out_path = results_path + out_fname

# -------------------
# Run + write alternative model
# -------------------
alt_df = run_meta_regression_with_predictors(
    input_df=input_df,
    predictors=alt_predictors,
    desc="Meta-regression (NO pLoF_ind, NO missense_ind)",
)

alt_df.to_csv(out_path, index=False, sep="\t")
print(f"Wrote: {out_path}  (rows={len(alt_df):,})")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# ============================================================
# Separate 2x2 heatmap grid (4 groups) for each comparison:
#   (1) FULL vs MINUS log_constraint      (HMM constraint)
#   (2) FULL vs MINUS GERP_RS
#   (3) FULL vs MINUS log_pathogenicity  (AM pathogenicity)
#   (4) FULL vs ALT (no pLoF/missense indicators)
# ============================================================

# -------------------
# SETTINGS
# -------------------
min_variants = 25
num_bins = 10

# Optional reference lines
DRAW_EXOME_LINES = False
exome_wide_threshold = 3.4e-7
exome_line = -np.log10(exome_wide_threshold)

# Annotation controls
USE_BINNED_ANNOTATION = False  # True=binned stacked labels, False=true (x,y) centered
ANNOT_FONTSIZE = 6
p_threshold = 1e-6
delta_threshold = 1

key_cols = ["gene_id", "gene_name", "group"]

# -------------------
# Load FULL model (y-axis)
# -------------------
full_df = pd.read_csv(results_path + "epilepsy_unified_model_pvalues.tsv", sep="\t")
full_df["p_unified"] = pd.to_numeric(full_df["p_unified"], errors="coerce")

# -------------------
# Comparison specs: (title, other_path, x_label, out_png)
#   x-axis = "without" model, y-axis = FULL
# -------------------
comparisons = [
    ("Unified Model p-values: with vs without HMM Constraint",
     results_path + "epilepsy_unified_model_pvalues_MINUS_log_constraint.tsv",
     r"$-log_{10}(p)$ without HMM constraint",
     results_path + "Figure_heatmap_FULL_vs_MINUS_log_constraint_2x2.png"),

    ("Unified Model p-values: with vs without GERP RS score",
     results_path + "epilepsy_unified_model_pvalues_MINUS_GERP_RS.tsv",
     r"$-log_{10}(p)$ without GERP RS",
     results_path + "Figure_heatmap_FULL_vs_MINUS_GERP_RS_2x2.png"),

    ("Unified Model p-values: with vs without AM Pathogenicity",
     results_path + "epilepsy_unified_model_pvalues_MINUS_log_pathogenicity.tsv",
     r"$-log_{10}(p)$ without AM pathogenicity",
     results_path + "Figure_heatmap_FULL_vs_MINUS_log_pathogenicity_2x2.png"),

    ("Unified Model p-values: with vs without pLoF/Missense Annotations",
     results_path + "epilepsy_unified_model_pvalues_NO_pLoF_NO_missense.tsv",
     r"$-log_{10}(p)$ without pLoF/missense",
     results_path + "Figure_heatmap_FULL_vs_ALT_no_pLoF_no_missense_2x2.png"),
]

def make_cmp(full_df: pd.DataFrame, other_df: pd.DataFrame) -> pd.DataFrame:
    other_df = other_df.copy()
    other_df["p_unified"] = pd.to_numeric(other_df["p_unified"], errors="coerce")

    cmp = (
        full_df[key_cols + ["n_variants", "p_unified"]]
        .rename(columns={"n_variants": "n_with", "p_unified": "p_with"})
        .merge(
            other_df[key_cols + ["n_variants", "p_unified"]]
            .rename(columns={"n_variants": "n_without", "p_unified": "p_without"}),
            on=key_cols,
            how="inner",
        )
    )

    cmp = cmp[(cmp["n_with"] >= min_variants) & (cmp["n_without"] >= min_variants)].copy()
    cmp["p_with"] = pd.to_numeric(cmp["p_with"], errors="coerce")
    cmp["p_without"] = pd.to_numeric(cmp["p_without"], errors="coerce")
    cmp = cmp.dropna(subset=["p_with", "p_without"]).reset_index(drop=True)

    eps = 1e-300
    cmp["log10_with"] = -np.log10(np.clip(cmp["p_with"].to_numpy(dtype=float), eps, 1.0))       # y (FULL)
    cmp["log10_without"] = -np.log10(np.clip(cmp["p_without"].to_numpy(dtype=float), eps, 1.0)) # x (WITHOUT)
    cmp["delta"] = cmp["log10_with"] - cmp["log10_without"]
    return cmp

def plot_heatmap_grid_no_gap(
    cmp: pd.DataFrame,
    title: str,
    x_label: str,
    out_png: str,
):
    groups = sorted(cmp["group"].dropna().unique())
    if len(groups) != 4:
        print(f"Warning: expected 4 groups, found {len(groups)}: {groups}")

    global_max = max(cmp["log10_with"].max(), cmp["log10_without"].max())
    if not np.isfinite(global_max):
        global_max = 1
    limit = global_max + 1

    x_edges = np.linspace(0, limit, num_bins + 1)
    y_edges = np.linspace(0, limit, num_bins + 1)

    # Manual 2x2 layout (no gaps), with shared colorbar
    fig = plt.figure(figsize=(12, 10))

    # Grid region; leave room for colorbar
    L, R = 0.08, 0.88
    B, T = 0.08, 0.92
    W, H = (R - L), (T - B)

    side = min(W / 2, H / 2)
    grid_W, grid_H = 2 * side, 2 * side
    grid_L = L + (W - grid_W) / 2
    grid_B = B + (H - grid_H) / 2

    positions = {
        0: [grid_L + 0 * side, grid_B + 1 * side, side, side],
        1: [grid_L + 1 * side, grid_B + 1 * side, side, side],
        2: [grid_L + 0 * side, grid_B + 0 * side, side, side],
        3: [grid_L + 1 * side, grid_B + 0 * side, side, side],
    }
    axes = [fig.add_axes(positions[i]) for i in range(4)]
    mappable = None

    for ax, group in zip(axes, groups):
        df = cmp[cmp["group"] == group].copy()

        h = ax.hist2d(
            df["log10_without"],   # x
            df["log10_with"],      # y
            bins=[x_edges, y_edges],
            cmap="Blues",
            norm=LogNorm()
        )
        mappable = h[3]

        # Diagonal
        ax.plot([0, limit], [0, limit], linestyle="--", color="red", linewidth=1.0, alpha=0.6)

        if DRAW_EXOME_LINES:
            ax.axhline(exome_line, linestyle=":", color="red", linewidth=1.2)
            ax.axvline(exome_line, linestyle=":", color="red", linewidth=1.2)

        # Annotation selection
        annot_df = df[
            ((df["p_with"] < p_threshold) | (df["p_without"] < p_threshold)) &
            (np.abs(df["delta"]) >= delta_threshold)
        ].copy()

        if USE_BINNED_ANNOTATION:
            annot_df["x_bin"] = np.digitize(annot_df["log10_without"], bins=x_edges) - 1
            annot_df["y_bin"] = np.digitize(annot_df["log10_with"], bins=y_edges) - 1

            x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])
            y_centers = 0.5 * (y_edges[:-1] + y_edges[1:])
            stack_offset = limit * 0.02

            for (x_idx, y_idx), gdf in annot_df.groupby(["x_bin", "y_bin"]):
                if 0 <= x_idx < len(x_centers) and 0 <= y_idx < len(y_centers):
                    x_center = x_centers[x_idx]
                    y_center = y_centers[y_idx]
                    total_h = (len(gdf) - 1) * stack_offset

                    gdf_sorted = gdf.sort_values("log10_with")
                    for i, (_, row) in enumerate(gdf_sorted.iterrows()):
                        ax.text(
                            x_center,
                            y_center - total_h / 2 + i * stack_offset,
                            row["gene_name"],
                            fontsize=ANNOT_FONTSIZE,
                            ha="center"
                        )
        else:
            for _, row in annot_df.iterrows():
                ax.text(
                    row["log10_without"],
                    row["log10_with"],
                    row["gene_name"],
                    fontsize=ANNOT_FONTSIZE,
                    ha="center",
                    va="center"
                )

        ax.set_xlim(0, limit)
        ax.set_ylim(0, limit)

        # Group label bottom-right
        ax.text(
            0.97 * limit, 0.03 * limit,
            f"Group: {group}",
            fontsize=12, fontweight="bold",
            ha="right", va="bottom"
        )

    # Remove interior tick labels
    axes[0].set_xticklabels([])
    axes[1].set_xticklabels([])
    axes[1].set_yticklabels([])
    axes[3].set_yticklabels([])

    fig.suptitle(title, fontsize=16, fontweight="bold", y=0.97)
    fig.supxlabel(x_label, fontsize=14)
    fig.supylabel(r"$-log_{10}(p)$ full model", fontsize=14)

    # External colorbar aligned to grid height
    cbar_ax = fig.add_axes([0.91, grid_B, 0.02, grid_H])
    cbar = fig.colorbar(mappable, cax=cbar_ax)
    cbar.set_label("Log-scaled count")

    plt.savefig(out_png, dpi=300, bbox_inches="tight")
    plt.show()


# -------------------
# Run all 4 plots (each as its own 2x2 figure)
# -------------------
for title, other_path, x_label, out_png in comparisons:
    other_df = pd.read_csv(other_path, sep="\t")
    cmp = make_cmp(full_df, other_df)
    plot_heatmap_grid_no_gap(cmp=cmp, title=title, x_label=x_label, out_png=out_png)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------
# Settings
# -------------------
min_variants = 25
exome_wide_threshold = 3.4e-7
exome_line = -np.log10(exome_wide_threshold)

p_threshold = 1e-6
delta_threshold = 1
x_offset = 0.15

key_cols = ["gene_id", "gene_name", "group"]

# -------------------
# Use same 4 comparisons as the heatmap cell
# (y = FULL, x = WITHOUT)
# -------------------
comparisons = [
    ("HMM Constraint", results_path + "epilepsy_unified_model_pvalues_MINUS_log_constraint.tsv"),
    ("GERP RS", results_path + "epilepsy_unified_model_pvalues_MINUS_GERP_RS.tsv"),
    ("AM Pathogenicity", results_path + "epilepsy_unified_model_pvalues_MINUS_log_pathogenicity.tsv"),
    ("pLoF/Missense Annotations", results_path + "epilepsy_unified_model_pvalues_NO_pLoF_NO_missense.tsv"),
]

# -------------------
# Load FULL model (y-axis)
# -------------------
# (If already in memory from the heatmap cell, reuse it.)
if "full_df" not in globals() or not isinstance(full_df, pd.DataFrame) or len(full_df) == 0:
    full_df = pd.read_csv(results_path + "epilepsy_unified_model_pvalues.tsv", sep="\t")
full_df["p_unified"] = pd.to_numeric(full_df["p_unified"], errors="coerce")

# -------------------
# Build long DF across the 4 comparisons
# -------------------
cmp_long = []

for label, other_path in comparisons:
    other_df = pd.read_csv(other_path, sep="\t")
    other_df["p_unified"] = pd.to_numeric(other_df["p_unified"], errors="coerce")

    cmp = (
        full_df[key_cols + ["n_variants", "p_unified"]]
        .rename(columns={"n_variants": "n_full", "p_unified": "p_full"})
        .merge(
            other_df[key_cols + ["n_variants", "p_unified"]]
            .rename(columns={"n_variants": "n_without", "p_unified": "p_without"}),
            on=key_cols,
            how="inner",
        )
    )

    cmp = cmp[(cmp["n_full"] >= min_variants) & (cmp["n_without"] >= min_variants)].copy()
    cmp["p_full"] = pd.to_numeric(cmp["p_full"], errors="coerce")
    cmp["p_without"] = pd.to_numeric(cmp["p_without"], errors="coerce")
    cmp = cmp.dropna(subset=["p_full", "p_without"]).reset_index(drop=True)

    eps = 1e-300
    cmp["log10_full"] = -np.log10(np.clip(cmp["p_full"].to_numpy(dtype=float), eps, 1.0))       # y
    cmp["log10_without"] = -np.log10(np.clip(cmp["p_without"].to_numpy(dtype=float), eps, 1.0)) # x
    cmp["delta"] = cmp["log10_full"] - cmp["log10_without"]
    cmp["comparison"] = label

    cmp_long.append(cmp)

cmp_long = pd.concat(cmp_long, ignore_index=True)

# -------------------
# Plot
# -------------------
plt.figure(figsize=(8, 6))

color_map = {label: None for label, _ in comparisons}

for label, _ in comparisons:
    d = cmp_long[cmp_long["comparison"] == label]
    sc = plt.scatter(
        d["log10_without"],
        d["log10_full"],
        s=10,
        alpha=0.22,
        label=label,
    )
    if color_map[label] is None:
        color_map[label] = sc.get_facecolors()[0]

# Limits (square, +1 padding like your prior scatter)
mx = float(max(cmp_long["log10_full"].max(), cmp_long["log10_without"].max()))
xmax = mx + 1
ymax = mx + 1
plt.xlim(0, xmax)
plt.ylim(0, ymax)

# 45-degree line (dimmer blue dashed)
plt.plot([0, xmax], [0, ymax], linestyle="--", color="blue", linewidth=1.0, alpha=0.6)

# Exome-wide threshold lines (red)
plt.axhline(exome_line, linestyle=":", color="red", linewidth=1.2)
plt.axvline(exome_line, linestyle=":", color="red", linewidth=1.2)

# -------------------
# Highlight + annotate: p < p_threshold in either model AND |delta| >= delta_threshold
# -------------------
for label, _ in comparisons:
    d = cmp_long[cmp_long["comparison"] == label].copy()
    sel = ((d["p_full"] < p_threshold) | (d["p_without"] < p_threshold)) & (np.abs(d["delta"]) >= delta_threshold)
    if sel.sum() == 0:
        continue

    plt.scatter(
        d.loc[sel, "log10_without"],
        d.loc[sel, "log10_full"],
        s=28,
        color=color_map[label],
        zorder=3
    )

    for _, row in d.loc[sel].iterrows():
        plt.text(
            row["log10_without"] + x_offset,
            row["log10_full"],
            row["gene_name"],
            fontsize=8,
            color="black",
            fontweight="medium"
        )

plt.xlabel(r"$-log_{10}(p)$ model WITHOUT specified annotation(s)", fontsize=12)
plt.ylabel(r"$-log_{10}(p)$ FULL model", fontsize=12)

plt.title(
    "Unified model p-values: FULL vs models without selected annotations",
    fontsize=14,
    fontweight="bold"
)

plt.legend(frameon=False, fontsize=9, loc="lower right")
plt.tight_layout()

plt.savefig(
    results_path + "Figure_scatter_FULL_vs_selected_annotation_removals.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()